# 🔧 AICE Associate - Data Preprocessing

AICE 시험에서 자주 나오는 전처리 패턴들입니다.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

# 샘플 데이터 로딩
df = pd.read_csv('../mock_exams/data/mock_22_telecom.csv')
print(f"Shape: {df.shape}")
df.head()

## 1. 결측치 처리

In [ ]:
# 결측치 확인
print("=== 결측치 확인 ===")
print(df.isna().sum())
print(f"\n총 결측치: {df.isna().sum().sum()}")

In [ ]:
# 결측치 처리
df_clean = df.copy()

# 수치형: 평균값
for col in df_clean.select_dtypes(include=[np.number]).columns:
    if df_clean[col].isna().sum() > 0:
        df_clean[col].fillna(df_clean[col].mean(), inplace=True)

# 범주형: 최빈값
for col in df_clean.select_dtypes(include=['object']).columns:
    if df_clean[col].isna().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

print("결측치 처리 완료!")
print(df_clean.isna().sum().sum())

## 2. 이상치 확인

In [ ]:
# 박스플롯으로 이상치 확인
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns[:4]

fig, axes = plt.subplots(1, len(numeric_cols), figsize=(15, 4))
for i, col in enumerate(numeric_cols):
    sns.boxplot(y=df_clean[col], ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# IQR 방식 이상치 탐지
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return len(outliers)

print("=== 이상치 개수 ===")
for col in df_clean.select_dtypes(include=[np.number]).columns:
    n_outliers = detect_outliers_iqr(df_clean, col)
    if n_outliers > 0:
        print(f"{col}: {n_outliers}개")

## 3. 인코딩

### 3-1. One-Hot Encoding (권장)

In [ ]:
# 범주형 컬럼 확인
cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
print(f"범주형 컬럼: {cat_cols}")

In [ ]:
# One-Hot Encoding
df_encoded = pd.get_dummies(df_clean, columns=cat_cols)

print(f"인코딩 전: {df_clean.shape}")
print(f"인코딩 후: {df_encoded.shape}")
df_encoded.head()

### 3-2. Label Encoding

In [ ]:
# Label Encoding 예시
le = LabelEncoder()
df_label = df_clean.copy()

for col in cat_cols:
    df_label[col + '_encoded'] = le.fit_transform(df_label[col])

df_label.head()

## 4. 스케일링

### 4-1. StandardScaler (표준화)

In [ ]:
# 수치형 컬럼만 선택
numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns
print(f"수치형 컬럼 수: {len(numeric_cols)}")

In [ ]:
# StandardScaler: 평균=0, 표준편차=1
scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

print("표준화 후 통계:")
df_scaled[numeric_cols[:3]].describe()

### 4-2. MinMaxScaler (정규화)

In [ ]:
# MinMaxScaler: 0~1 범위
scaler = MinMaxScaler()
df_normalized = df_encoded.copy()
df_normalized[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

print("정규화 후 통계:")
df_normalized[numeric_cols[:3]].describe()

## 5. 파생변수 생성

In [ ]:
# 예시: 비율 변수
df_feature = df_clean.copy()

# 총 요금 대비 월 요금 비율
df_feature['monthly_ratio'] = df_feature['monthly_charges'] / (df_feature['total_charges'] + 1)

df_feature[['monthly_charges', 'total_charges', 'monthly_ratio']].head()

In [ ]:
# 가입기간 구간화
df_feature['tenure_group'] = pd.cut(
    df_feature['tenure'], 
    bins=[0, 12, 24, 48, 72],
    labels=['0-12', '12-24', '24-48', '48+']
)

df_feature['tenure_group'].value_counts()

## 6. 피처/타겟 분리

In [ ]:
# 타겟 컬럼 확인
print(f"타겟: churn")
print(df_encoded['churn'].value_counts())

In [ ]:
# 피처/타겟 분리
drop_cols = ['churn', 'customer_id']  # 타겟 + ID
drop_cols = [col for col in drop_cols if col in df_encoded.columns]

X = df_encoded.drop(drop_cols, axis=1)
y = df_encoded['churn']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\n피처 목록:")
print(X.columns.tolist())

## 7. Train/Test 분할

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

---

## ✅ 전처리 체크리스트

| 단계 | 확인 사항 |
|------|----------|
| 1. 결측치 | `df.isna().sum()` 으로 확인 후 처리 |
| 2. 이상치 | 박스플롯 또는 IQR로 확인 |
| 3. 인코딩 | 범주형 → One-Hot 또는 Label Encoding |
| 4. 스케일링 | StandardScaler 또는 MinMaxScaler |
| 5. 파생변수 | 도메인 지식 활용 |
| 6. 분리 | X, y 분리 → train_test_split |

**시험에서는 변수명을 정확히 지켜야 합니다!** ⚠️